# 19 · GoalScope out of domain

Does the goal-head read-out (GoalScope) surface a coherent *instrumental goal* when the model is
doing something **other than writing code**? Two operationalizations:

1. **Function-vector tasks (Todd et al.)** — few-shot prompts (`antonym`, `country-capital`, …). The
   model keeps performing the task; can GoalScope verbalize *what task it is doing* ("mapping a
   country to its capital", "getting the antonym of X")? Most scalable — 40+ tasks in `data/FV_dataset`.
2. **Planning (non-code)** — cooking recipe, trip plan, short story. Can it read out the *next step*?
   (Prototype: `notebooks/clock/__non_code.ipynb`.)

Flow: pick a **template prompt** → let the LM generate ~256 tokens → **apply GoalScope** over the
generated span, with a **selectable read-out prompt** (`GOALSCOPE_PROMPTS`). Qualitative first; scaling
notes at the bottom.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, json, random, logging
from pathlib import Path
from glob import glob

# pick free GPUs (32B fits on one A100-80GB; 2x smaller cards also OK)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4,5")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")

In [2]:
import torch
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
# MODEL_NAME = "allenai/Olmo-3.1-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME, dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

2026-09-12 09:55:57 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-09-12 09:55:57 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 09:55:57 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-09-12 09:55:57 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 09:55:57 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-09-12 09:55:57 httpx INFO     HTTP Re

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-09-12 09:56:21 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 09:56:21 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-09-12 09:56:22 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40


## Goal heads

Load the frozen goal-head set (git-tracked `head_sets/<model>/goal__lam20_thr0.9_band*.json`, same
set the goalscope evals use). Falls back to the trained mask checkpoint if no JSON is present.

In [4]:
slug = MODEL_NAME.split("/")[-1]
_json = sorted(glob(f"../../head_sets/{slug}/goal__lam20_thr0.9_band*.json"))
if _json:
    goal_heads = [tuple(x) for x in json.load(open(_json[0]))]
    print(f"goal heads from {Path(_json[0]).name}: {len(goal_heads)}")
else:  # fall back to the mask checkpoint (see __non_code.ipynb)
    from src.goals.head_loc_attn import HeadMask
    from src.utils import env_utils
    ckpt = torch.load(os.path.join(env_utils.DEFAULT_RESULTS_DIR,
                      f"localization/{slug}/lam20/head_mask.pt"), map_location="cpu")
    LAYERS = ckpt["layers"]
    mask = HeadMask(len(LAYERS), ckpt["mask_values"].shape[1])
    with torch.no_grad():
        mask.values.copy_(ckpt["mask_values"])
        band = {"Qwen2.5-Coder-32B-Instruct": (25, 56)}.get(slug)
        if band:
            mask.values[:band[0]] = 0; mask.values[band[1]:] = 0
    goal_heads = mask.selected_heads(LAYERS, threshold=0.9)
    print(f"goal heads from mask ckpt: {len(goal_heads)}")

goal heads from goal__lam20_thr0.9_band25-56.json: 122


## Read-out prompt registry — *choose the GoalScope setup*

`apply_goalscope(..., goalscope_prompt=<str>)` sets the destination prompt the goal signal is steered
into. The built-in one is code-flavoured ("What are you doing at this line of the code"); for
non-code we want task/plan-flavoured verbalizers. Pick one by key in `run_case(..., gs_key=...)`.

In [5]:
from src.goals.goalscope import _build_goalscope_prompt

def _chat(system, user, assistant=""):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    return mt.tokenizer.apply_chat_template(
        msgs, add_generation_prompt=False, continue_final_message=True, tokenize=False)

_ASSIST = "You are a helpful assistant."

GOALSCOPE_PROMPTS = {
    # code-flavoured default (baseline / sanity — expect it to fit code, not tasks)
    "code_default": _build_goalscope_prompt(mt),
    # function-vector: verbalize the hidden rule / task being performed
    "task_rule":   _chat(_ASSIST, "You are answering a series of questions that all follow the same "
                         "hidden rule. In one short phrase, what task are you performing?"),
    "task_io":     _chat(_ASSIST, "You are mapping each input to an output by a fixed rule. Describe "
                         "the rule in one short phrase: 'I am mapping X to ...'"),
    # planning: verbalize the next step
    "next_step":   _chat(_ASSIST, "You are in the middle of writing a response. What is the next step?"),
    "recipe_next": _chat("You are a chef who shares recipes on your blog.",
                         "You are in the middle of writing a recipe. What is the next step?"),
    "trip_next":   _chat("You are a travel expert who helps people plan trips.",
                         "You are in the middle of writing a travel plan. What is the next step?"),
    "story_next":  _chat("You are a short story writer.",
                         "You are in the middle of writing a short story. What should happen next?"),
}
print(list(GOALSCOPE_PROMPTS))

['code_default', 'task_rule', 'task_io', 'next_step', 'recipe_next', 'trip_next', 'story_next']


## `run_case` — generate, then scope the generated span

Generates ~256 tokens from `task_prompt`, then applies GoalScope to **every generated token** (passing a pre-tokenized sequence + `token_positions` so positions align exactly). Each scoped position runs its own short read-out generation, so a full 256-token span is heavy on 32B — trim `gen_tokens` for a quick first look.

In [6]:
from src.functional import generate_with_patch
from src.tokens import prepare_input
from src.goals.goalscope import apply_goalscope

@torch.no_grad()
def generate_clean(task_prompt, max_new_tokens=256):
    tok = prepare_input(task_prompt, mt, padding_side="left")
    return generate_with_patch(mt, inputs=tok, n_gen_per_prompt=1,
                               max_new_tokens=max_new_tokens, do_sample=False,
                               remove_prefix=True)[0]

@torch.no_grad()
def run_case(task_prompt, gs_key, *, gen=None, gen_tokens=256,
             strategy="additive_norm", alpha=0.1, steer_layer_range=(35, 46),
             max_new_tokens=48, heads=None, save_path=None, show=True):
    """Generate ~gen_tokens from task_prompt, then GoalScope EVERY generated token.
    task_prompt: the prompt to continue (chat OR raw). gs_key: key in GOALSCOPE_PROMPTS."""
    heads = heads or goal_heads
    if gen is None:
        gen = generate_clean(task_prompt, gen_tokens)
    rich_print(f"[bold]clean[/bold] [white]{escape(task_prompt)}[/white]"
               f"[sea_green3]{escape(gen)}[/sea_green3]\n")
    enc = prepare_input(task_prompt + gen, mt, padding_side="left")
    p0  = prepare_input(task_prompt, mt, padding_side="left")["input_ids"].shape[1]
    ptot = enc["input_ids"].shape[1]
    positions = list(range(p0, ptot))                   # ALL generated tokens
    results = apply_goalscope(
        prompt=enc, mt=mt, selected_heads=heads,
        strategy=strategy, alpha=alpha, steer_layer_range=steer_layer_range,
        max_new_tokens=max_new_tokens, n_steer_tokens="all",
        goalscope_prompt=GOALSCOPE_PROMPTS[gs_key], token_positions=positions,
        include_noncode=True, title=f"GoalScope · {gs_key} · a={alpha}",
        show=show, save_path=save_path)
    return gen, results

## A · Function-vector tasks (Todd et al.)

Build a few-shot prompt and ask the model to **continue the list** (so it keeps performing the task
across many tokens). Read-out with `gs_key="task_rule"` / `"task_io"` — does it verbalize the mapping?

In [7]:
FV_DIR = "../../data/FV_dataset/abstractive"

def build_fv_prompt(task, n_shots=12, seed=0):
    """Raw few-shot completion (NO chat scaffolding): plain Q/A lines; the model just continues."""
    data = json.load(open(f"{FV_DIR}/{task}.json"))
    rng = random.Random(seed)
    pick = rng.sample(data, n_shots + 1)
    shots, query = pick[:n_shots], pick[n_shots]
    body = "\n".join(f"Q: {p['input']}\nA: {p['output']}" for p in shots)
    prompt = body + f"\nQ: {query['input']}\nA:"       # let the LM continue the list
    return prompt, query

print([Path(f).stem for f in sorted(glob(f"{FV_DIR}/*.json"))])

['ag_news', 'antonym', 'capitalize', 'capitalize_first_letter', 'capitalize_last_letter', 'capitalize_second_letter', 'commonsense_qa', 'country-capital', 'country-currency', 'english-french', 'english-german', 'english-spanish', 'landmark-country', 'lowercase_first_letter', 'lowercase_last_letter', 'national_parks', 'next_capital_letter', 'next_item', 'park-country', 'person-instrument', 'person-occupation', 'person-sport', 'present-past', 'prev_item', 'product-company', 'sentiment', 'singular-plural', 'synonym', 'word_length']


In [8]:
# antonym: expect read-out like "opposite / antonym of the word"
p, q = build_fv_prompt("antonym", n_shots=6, seed=1)
# p, q = build_fv_prompt("present-past", n_shots=6, seed=1)
# p, q = build_fv_prompt("next_item", n_shots=6, seed=1)
gen, _ = run_case(p, gs_key="task_rule", gen_tokens=64, alpha=0.1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


clean Q: enter
A: exit
Q: feasible
A: infeasible
Q: manned
A: unmanned
Q: fuzzy
A: smooth
Q: arise
A: settle
Q: far
A: near
Q: mercy
A: cruelty
Q: light
A: heavy
Q: fast
A: slow
Q: high
A: low
Q: open
A: closed
Q: wet
A: dry
Q: hot
A: cold
Q: bright
A: dim
Q: loud
A:

GoalScope[additive_norm]:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
# country-capital: expect "mapping a country to its capital"
# p, q = build_fv_prompt("country-capital", n_shots=6, seed=1)
# p, q = build_fv_prompt("person-occupation", n_shots=6, seed=1)
# p, q = build_fv_prompt("person-instrument", n_shots=6, seed=1)
p, q = build_fv_prompt("sentiment", n_shots=3, seed=1)
gen, _ = run_case(p, gs_key="task_rule", gen_tokens=64, alpha=0.1)

clean Q: Very predictable but still entertaining
A: positive
Q: The entire movie is in need of a scented bath.
A: negative
Q: A manically generous Christmas vaudeville.
A: positive
Q: Lacks depth.
A: negative
Q: It's a good movie, but not great.
A: mixed
Q: A film that is both a visual and auditory delight.
A: positive
Q: It's a movie that will make you laugh and cry.
A: positive
Q: It's a movie that will make you feel

GoalScope[additive_norm]:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
# # quick contrast: same generation, code-flavoured read-out prompt (sanity — should fit worse)
# run_case(p, gs_key="code_default", gen=gen, alpha=0.1)

### Causality of FV setup with goal heads

In [12]:
# ── FV causality setup: precompute the task mean head-activation ā (SEPARATE from the patched run) ──
# ā[task] = mean o_proj INPUT per layer over MEAN_TRIALS clean ICL prompts (Todd et al. function vector,
# in per-head-input form). Both intervention strategies below derive their injection from this same ā.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import baukit
from rich.table import Table
from scripts.fv.fv_common import (load_fv_task, build_icl_prompt,
                                  first_answer_token_id, compute_mean_head_activations)
from src.goals.head_loc_attn import HeadMask, forward_with_src_replacement
from src.functional import PatchSpec, patch_with_baukit
from src.tokens import prepare_input

N_LAYER = mt.n_layer
N_HEADS = mt.model.config.num_attention_heads
HD      = mt.head_dim
FV_LAYERS = list(range(N_LAYER))

fv_heads = [tuple(x) for x in json.load(open(f"../../head_sets/{slug}/fv__cma_top150.json"))]
REC_SETS = {"FV": fv_heads, "goal": goal_heads}
print(f"goal heads {len(goal_heads)} · FV heads {len(fv_heads)}")

FV_TASKS    = ["antonym", "present-past", "singular-plural", "country-capital", "landmark-country", "sentiment"]
N_QUERY     = 32          # zero-shot queries per task
MEAN_TRIALS = 32          # clean ICL prompts averaged for ā per task

# --- precompute ā once per task (the expensive part; independent of strategy/alpha/head set) ---
A_BAR, TASKS = {}, {}
for t in FV_TASKS:
    TASKS[t] = load_fv_task(t, seed=42)
    A_BAR[t] = compute_mean_head_activations(mt, TASKS[t], FV_LAYERS, n_shots=10,
                                             n_trials=MEAN_TRIALS, batch_size=16, seed=42)  # (L, n_embd)
    logger.info(f"ā[{t}] {tuple(A_BAR[t].shape)}")

# --- injection helpers: the SAME interventions GoalScope uses, but returning 1-token logits ---
def _binmask(heads):
    m = HeadMask(N_LAYER, N_HEADS)
    with torch.no_grad():
        m.values.zero_()
        for (l, h) in heads: m.values[l, h] = 1.0
    return m

def _agg_write(a_bar, heads):
    """Aggregated residual write of `heads` from ā: Σ_(l,h) W_o[l][:, h] @ ā[l][h-slice]  (n_embd,).
    Mirrors collect_head_steering_vecs, but from the cached mean instead of a live forward."""
    g = torch.zeros(mt.n_embd, dtype=torch.float32)
    for (l, h) in heads:
        Wo = baukit.get_module(mt._model, mt.attn_module_name_format.format(l) + ".o_proj").weight
        sl = slice(h * HD, (h + 1) * HD)
        g += (Wo[:, sl].float().cpu() @ a_bar[l][sl].float().cpu())
    return g

def patched_logits(inp, a_bar, heads, mask, strategy, alpha, steer_layer_range=(35, 46)):
    """1-token patched logits under the chosen GoalScope intervention at the LAST position.
      strategy='replace'       -> per-head o_proj-input replacement at each head's layer
                                   (forward_with_src_replacement; α=1.0 = pure swap).
      strategy='additive_norm' -> aggregate `heads` into one write vec and add it norm-relative
                                   into the residual over steer_layer_range (PatchSpec add_norm)."""
    if strategy == "replace":
        pl = forward_with_src_replacement(mt, inp, a_bar, mask, layers=FV_LAYERS,
                                          alpha=alpha, patch_positions=[-1]).float()
        return pl if pl.ndim == 2 else pl.unsqueeze(0)
    elif strategy == "additive_norm":
        g = _agg_write(a_bar, heads)
        g = (alpha * g / g.norm().clamp_min(1e-6))                       # α-scaled UNIT dir; add_norm supplies ||h||
        specs = [PatchSpec(location=(mt.layer_name_format.format(l), -1), patch=g,
                           strategy="add_norm", on_pass="all") for l in range(*steer_layer_range)]
        out = patch_with_baukit(mt, inp, specs, {})
        return out.logits[:, -1, :].float()
    raise ValueError(strategy)

rec_masks = {name: _binmask(hs) for name, hs in REC_SETS.items()}
def zero_shot(x): return build_icl_prompt([], x)                          # "\n\nQ: x\nA:"
print("ā precomputed ·", list(A_BAR), "· helpers ready (patched_logits, _agg_write)")

goal heads 122 · FV heads 150
2026-09-12 10:23:06 __main__ INFO     ā[antonym] (64, 5120)
2026-09-12 10:23:11 __main__ INFO     ā[present-past] (64, 5120)
2026-09-12 10:23:15 __main__ INFO     ā[singular-plural] (64, 5120)
2026-09-12 10:23:21 __main__ INFO     ā[country-capital] (64, 5120)
2026-09-12 10:23:26 __main__ INFO     ā[landmark-country] (64, 5120)
2026-09-12 10:23:33 __main__ INFO     ā[sentiment] (64, 5120)
ā precomputed · ['antonym', 'present-past', 'singular-plural', 'country-capital', 'landmark-country', 'sentiment'] · helpers ready (patched_logits, _agg_write)


In [13]:
# ── FV recovery — CHOOSE strategy + alpha; 1-token evaluation (top-1 / dP on the answer token) ──
STRATEGY  = "additive_norm"      # "replace" (α=1.0 = pure head swap) | "additive_norm" (α≈0.1)
ALPHA     = 0.1                  # replace: 1.0 · additive_norm: ~0.1 (fraction of local residual norm)
STEER_BAND = (35, 46)            # additive_norm injection band (ignored for replace)

rec_rows = []                                                # one row per (task, head-set)
for t in FV_TASKS:
    task, a_bar = TASKS[t], A_BAR[t]
    qs = task.valid[:N_QUERY]
    clean_hit = 0
    agg = {name: dict(patch_hit=0, dp=0.0) for name in REC_SETS}
    for q in qs:
        prompt = zero_shot(q["input"])
        tid = first_answer_token_id(mt, prompt, q["output"])
        inp = prepare_input(prompts=prompt, tokenizer=mt)
        with torch.inference_mode():
            cl = mt._model(**dict(inp)).logits[:, -1, :].float()
            cp = torch.softmax(cl, -1)[0]
            clean_hit += int(cp.argmax().item() == tid)
            for name, heads in REC_SETS.items():             # same ā; only the head set / mask differs
                pl = patched_logits(inp, a_bar, heads, rec_masks[name], STRATEGY, ALPHA, STEER_BAND)
                pp = torch.softmax(pl, -1)[0]
                agg[name]["patch_hit"] += int(pp.argmax().item() == tid)
                agg[name]["dp"] += float(pp[tid].item() - cp[tid].item())
    for name in REC_SETS:
        rec_rows.append(dict(task=t, heads=name, n=len(qs), clean=clean_hit/len(qs),
                             patch=agg[name]["patch_hit"]/len(qs), dP=agg[name]["dp"]/len(qs)))
    logger.info(f"[{t}] clean {clean_hit}/{len(qs)}  " +
                "  ".join(f"{name} {agg[name]['patch_hit']}/{len(qs)}" for name in REC_SETS))

rec_df = pd.DataFrame(rec_rows)
tbl = Table(title=f"FV recovery — goal vs FV heads · strategy={STRATEGY} α={ALPHA} · zero-shot query, 1-token")
for c in ["task", "n", "clean top-1", "FV patch", "goal patch", "FV dP", "goal dP"]:
    tbl.add_column(c, justify=("left" if c == "task" else "right"))
for t in FV_TASKS:
    d = {r["heads"]: r for r in rec_rows if r["task"] == t}
    tbl.add_row(t, str(d["FV"]["n"]), f"{d['FV']['clean']:.2f}",
                f"{d['FV']['patch']:.2f}", f"{d['goal']['patch']:.2f}",
                f"{d['FV']['dP']:+.3f}", f"{d['goal']['dP']:+.3f}")
rich_print(tbl)

2026-09-12 10:25:25 __main__ INFO     [antonym] clean 2/32  FV 17/32  goal 14/32
2026-09-12 10:27:13 __main__ INFO     [present-past] clean 0/32  FV 26/32  goal 25/32
2026-09-12 10:29:01 __main__ INFO     [singular-plural] clean 2/32  FV 30/32  goal 32/32
2026-09-12 10:30:53 __main__ INFO     [country-capital] clean 3/32  FV 25/32  goal 16/32
2026-09-12 10:32:44 __main__ INFO     [landmark-country] clean 0/32  FV 17/32  goal 0/32
2026-09-12 10:34:35 __main__ INFO     [sentiment] clean 0/32  FV 18/32  goal 0/32


 FV recovery — goal vs FV heads · strategy=additive_norm α=0.1 · zero-shot query, 
                                     1-token                                      
┏━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ task             ┃  n ┃ clean top-1 ┃ FV patch ┃ goal patch ┃  FV dP ┃ goal dP ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ antonym          │ 32 │        0.06 │     0.53 │       0.44 │ +0.403 │  +0.352 │
│ present-past     │ 32 │        0.00 │     0.81 │       0.78 │ +0.757 │  +0.664 │
│ singular-plural  │ 32 │        0.06 │     0.94 │       1.00 │ +0.812 │  +0.916 │
│ country-capital  │ 32 │        0.09 │     0.78 │       0.50 │ +0.611 │  +0.362 │
│ landmark-country │ 32 │        0.00 │     0.53 │       0.00 │ +0.384 │  +0.001 │
│ sentiment        │ 32 │        0.00 │     0.56 │       0.00 │ +0.338 │  +0.004 │
└──────────────────┴────┴─────────────┴──────────┴────────────┴────────┴─────────┘

In [ ]:
# ── summary: mean over tasks + "goal recovers X% of the FV effect" + bar plot ──
from src.utils import env_utils
SET_COLOR = {"FV": "#55a868", "goal": "#dd8452"}
mean_clean = rec_df.groupby("task").clean.first().mean()
summ = rec_df.groupby("heads").agg(patch=("patch", "mean"), dP=("dP", "mean"))

st = Table(title=f"FV recovery — mean over {len(FV_TASKS)} tasks (clean top-1 = {mean_clean:.2f})")
for c in ["head set", "patch top-1", "mean dP(correct)"]:
    st.add_column(c, justify=("left" if c == "head set" else "right"))
for name in ["FV", "goal"]:
    st.add_row(name, f"{summ.loc[name, 'patch']:.2f}", f"{summ.loc[name, 'dP']:+.3f}")
rich_print(st)

fv_eff   = summ.loc["FV", "patch"] - mean_clean            # top-1 lift over clean
goal_eff = summ.loc["goal", "patch"] - mean_clean
frac_top1 = goal_eff / fv_eff if abs(fv_eff) > 1e-9 else float("nan")
frac_dp   = summ.loc["goal", "dP"] / summ.loc["FV", "dP"] if abs(summ.loc["FV", "dP"]) > 1e-9 else float("nan")
if frac_top1 == frac_top1:
    rich_print(f"[bold]goal recovers {frac_top1:.0%} of the FV top-1 effect[/bold] "
               f"({frac_dp:+.0%} on dP) — 0% = no FV effect, 100% = matches FV heads")
else:
    rich_print("[bold]FV arm shows no top-1 lift over clean — recovery ratio undefined[/bold]")

names = ["FV", "goal"]; xs = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vals = [summ.loc[n, "patch"] for n in names]
axes[0].bar(xs, vals, color=[SET_COLOR[n] for n in names], width=0.6)
axes[0].axhline(mean_clean, ls="--", c="grey", lw=1, label=f"clean = {mean_clean:.2f}")
for x, v in zip(xs, vals): axes[0].text(x, v, f"{v:.2f}", ha="center", va="bottom")
axes[0].set_xticks(xs); axes[0].set_xticklabels(names); axes[0].set_ylabel("patch top-1 accuracy")
axes[0].set_title("FV recovery — top-1"); axes[0].legend()
vals2 = [summ.loc[n, "dP"] for n in names]
axes[1].bar(xs, vals2, color=[SET_COLOR[n] for n in names], width=0.6)
axes[1].axhline(0, c="grey", lw=0.8)
for x, v in zip(xs, vals2): axes[1].text(x, v, f"{v:+.3f}", ha="center", va="bottom" if v >= 0 else "top")
axes[1].set_xticks(xs); axes[1].set_xticklabels(names); axes[1].set_ylabel("mean dP(correct)")
axes[1].set_title("FV recovery — dP")
plt.suptitle(f"do goal heads carry the function vector? (mean over {len(FV_TASKS)} tasks)")
plt.tight_layout()
FIGDIR = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "dissociation", slug)
os.makedirs(FIGDIR, exist_ok=True)
FIG = os.path.join(FIGDIR, "goalscope_ood_fv_recovery.png")
plt.savefig(FIG, dpi=110, bbox_inches="tight"); print("saved", FIG); plt.show()

## B · Planning (non-code)

Recipe / trip / story. Read-out with the plan-flavoured verbalizers — does it name the *next step*?
(Ports the `__non_code.ipynb` prompts.)

In [23]:
recipe = _chat("You are a chef who occasionally shares recipes on your blog.",
               "Give me a recipe for making pasta.")
gen, _ = run_case(recipe, gs_key="recipe_next", gen_tokens=256, alpha=0.1)

clean <|im_start|>system
You are a chef who occasionally shares recipes on your blog.<|im_end|>
<|im_start|>user
Give me a recipe for making pasta.<|im_end|>
<|im_start|>assistant
Certainly! Making pasta from scratch is a rewarding experience that allows you to control the quality and flavor of
your noodles. Here’s a simple recipe for classic egg pasta:

### Ingredients:
- 2 cups (250g) all-purpose flour, plus extra for dusting
- 3 large eggs
- A pinch of salt

### Equipment:
- Large mixing bowl
- Rolling pin
- Pasta cutter or sharp knife
- Boiling pot
- Colander

### Instructions:

#### Step 1: Prepare the Dough
1. **Mix Dry Ingredients**: In a large mixing bowl, combine the flour and salt. Make a well in the center of the 
flour mixture.
2. **Add Eggs**: Crack the eggs into the well and beat them lightly with a fork.
3. **Knead the Dough**: Gradually incorporate the flour into the eggs by stirring with a fork until the dough 
starts to come together. Then, turn it out onto a floured surface and knead for about 5-7 minutes until the dough 
is smooth and elastic. If the dough is too sticky, add a little more flour; if it's too dry, add a bit more water.

#### Step 2: Rest the Dough
1. **Cover and Rest

GoalScope[additive_norm]:   0%|          | 0/16 [00:00<?, ?it/s]

In [24]:
trip = _chat("You are a travel expert who helps people plan their trips.",
             "Give me a travel plan for a weekend trip to Rome, Italy.")
gen, _ = run_case(trip, gs_key="trip_next", gen_tokens=256, alpha=0.1)

clean <|im_start|>system
You are a travel expert who helps people plan their trips.<|im_end|>
<|im_start|>user
Give me a travel plan for a weekend trip to Rome, Italy.<|im_end|>
<|im_start|>assistant
Absolutely! Rome is a fantastic city with a rich history and vibrant culture, making it perfect for a weekend 
getaway. Here’s a detailed plan to help you make the most of your time there:

### Friday Night:
- **Arrival and Check-In:** Arrive in Rome and check into your hotel. If you're staying near the Colosseum or 
Piazza Navona, you'll be centrally located and close to many attractions.
- **Dinner:** Enjoy dinner at a traditional Roman trattoria. Try some classic Italian dishes like carbonara or osso
buco.

### Saturday Morning:
- **Colosseum and Roman Forum:** Start your day early to avoid crowds. The Colosseum is an iconic symbol of ancient
Rome, and the Roman Forum offers insights into daily life in the Roman Empire.
- **Palatine Hill:** After visiting the Colosseum, head to Palatine Hill for a view of the Roman Forum and the 
Circus Maximus. It's a great spot to understand the layout of ancient Rome.

### Saturday Afternoon:
- **Pantheon:** Walk over to the Pantheon, one of the best-preserved buildings from ancient Rome. Take a moment to 
admire its stunning dome.
- **Piazza Navona:**

GoalScope[additive_norm]:   0%|          | 0/16 [00:00<?, ?it/s]

## Notes · scaling brainstorm

Qualitative questions to judge from the panels above:
- Does the read-out name the **task/next-step** (not just echo surface tokens)? Is it stable across the span?
- Which `gs_key` works — does a task-flavoured prompt beat the code-flavoured `code_default`?
- Sensitivity to `alpha` / `steer_layer_range` / head set (122 vs 250 vs sparse).

If it holds, scale via the FV bank (auto-gradable):
- **Metric:** oracle-judge each read-out vs the known task label (`data/FV_dataset/<split>/<task>.json`
  name → gold description). Reuse `judge_implements`-style grading from `ablation_common.py`.
- **Coverage:** 40+ abstractive + extractive tasks × N seeds → a task-verbalization accuracy curve,
  analogous to the goalscope faithfulness skyline but for non-code tasks.
- **Control:** shuffle the goal→readout head map, or scope a no-task prompt, to bound chance.